In [ ]:
%pip install crunch-cli --upgrade --quiet --progress-bar off
!crunch setup-notebook structural-break-real-time lDiymeFUIVpfCDSUWoGsz7A0

Note: you may need to restart the kernel to use updated packages.
crunch-cli, version 11.8.0
delete /Users/minqi/Documents/ADIA_Lab_Structural_Break_Challenge/submissions/.crunchdao
main.py: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/submissions/67646/main.py (11725 bytes)
notebook.ipynb: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/submissions/67646/notebook.ipynb (24587 bytes)
requirements.txt: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/submissions/67646/requirements.original.txt (188 bytes)
data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_train.parquet (218514418 bytes)
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_test.reduced.parquet (2587435 bytes)
data/y_train.parquet: download from https:crunchdao--comp

In [ ]:
import crunch
# crunch_tools = crunch.load_notebook()   # uncomment for local test / submit

loaded inline runner with module: <module '__main__'>

cli version: 11.8.0
available ram: 8.00 gb
available cpu: 8 core
----


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
import os, math
from bisect import bisect_left, bisect_right
from collections import deque
from typing import Iterable, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb

In [13]:
# ---- helpers computed once per series on the reference ------------------
def _autocorr(x, lag):
    if len(x) <= lag:
        return 0.0
    a, b = x[:-lag], x[lag:]
    a = a - a.mean(); b = b - b.mean()
    d = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / d) if d > 0 else 0.0


def _fit_ar(x, p=5):
    """Least-squares AR(p) with intercept. Returns (coefs, resid_var)."""
    if len(x) <= p + 5:
        return None, 1.0
    rows = np.lib.stride_tricks.sliding_window_view(x, p + 1)
    Y = rows[:, -1]
    Xd = np.column_stack([np.ones(len(rows)), rows[:, :-1][:, ::-1]])
    beta, *_ = np.linalg.lstsq(Xd, Y, rcond=None)
    resid = Y - Xd @ beta
    return beta, float(resid.var()) if len(resid) else 1.0

In [14]:
# Streaming O(1) / O(window) feature extractor + LightGBM train/infer.
# Same StreamState is used to build training rows and to stream at inference,
# so the feature vectors match exactly. Pure numpy, no scipy.
#
# Timeout patches vs the previous submission (2026-07-14 run):
#   A. Force num_threads=1 on each booster before predict() — thread-pool
#      wake-up dominates single-row predict cost; ~5-20x speedup.
#   B. StreamState.update() writes into a preallocated ndarray and returns it,
#      instead of building a dict + doing a Python copy loop per point.
#   E. bisect on Python lists (faster than on numpy arrays for scalars);
#      cache 1/ref_n, 1/n_bins, sd_h, etc. as attributes.


class StreamState:
    """Per-series streaming feature extractor. reset() once, update(x) per point.

    update() returns a np.ndarray view of a preallocated buffer of length ncol.
    Caller must .copy() it if they want to keep it (train does; infer just writes
    it into the LGBM input row immediately).
    """

    VAR_MIN = 20  # guard: log-var features are unstable for tiny n

    # Base feature order — extended in __init__ with per-alpha / per-window blocks.
    _BASE = (
        "n_online", "mean_shift_z", "last_z", "online_logvar", "online_std",
        "online_skew", "online_kurt", "cumsum_range", "cusum_max",
        "tail2_frac", "tail3_frac", "jump_freq", "signrun_freq",
        "ks_stat", "rank_dev", "ac1_online", "ac1_shift", "ar_resid_logratio",
        "ref_log_n", "ref_skew", "ref_kurt", "ref_ac1",
    )

    def __init__(self, bins=16, roll_windows=(50, 200),
                 ewma_alphas=(0.05, 0.2), ar_p=5):
        self.bins = bins
        self.rw = tuple(roll_windows)
        self.alphas = tuple(ewma_alphas)
        self.p = ar_p
        self.feature_names = list(self._BASE)
        for a in self.alphas:
            self.feature_names += ["ewma%.2f_z" % a, "ewma%.2f_logvar" % a]
        for w in self.rw:
            self.feature_names += ["roll%d_meanshift" % w, "roll%d_logvar" % w]
        self.ncol = len(self.feature_names)
        self._ew_off = len(self._BASE) 
        self._roll_off = self._ew_off + 2 * len(self.alphas)
        self._out = np.zeros(self.ncol, dtype=np.float64)
        # bin edges shape (unchanged after reset)
        self._bin_targets = np.linspace(1.0 / self.bins, 1.0, self.bins)

    def reset(self, reference):
        r = np.asarray(reference, dtype=np.float64)
        self.mu_h = float(r.mean()) if len(r) else 0.0
        self.sd_h = max(float(r.std(ddof=1)) if len(r) > 1 else 1.0, 1e-8)
        self.var_h = self.sd_h ** 2
        self.ref_n = len(r)
        self._inv_refn = 1.0 / max(self.ref_n, 1)
        z = (r - self.mu_h) / self.sd_h if len(r) else r
        self.ref_skew = float((z ** 3).mean()) if len(r) else 0.0
        self.ref_kurt = float((z ** 4).mean() - 3.0) if len(r) else 0.0
        self.ref_ac1 = _autocorr(r, 1)
        self.ref_log_n = math.log(self.ref_n + 1.0)
        # Plain Python lists for cheap bisect on scalars.
        self.sorted_ref = np.sort(r).tolist() if len(r) else []
        edges = (np.quantile(r, np.linspace(0, 1, self.bins + 1)[1:-1])
                 if len(r) else np.zeros(self.bins - 1))
        self.edges = edges.tolist()
        self.ar, self.ar_var_h = _fit_ar(r, self.p)
        # streaming accumulators
        self.n = 0
        self.mean = self.M2 = self.M3 = self.M4 = 0.0
        self.sx = self.sxx = self.sxy = 0.0
        self.cmax = self.cmin = self.cumsum = 0.0
        self.cusum_p = self.cusum_n = self.cusum_max = 0.0
        self.tail2 = self.tail3 = 0
        self.binc = np.zeros(self.bins)
        self.rank_sum = 0.0
        self.jumps = 0; self.signruns = 0; self.prev = None
        self.ar_resid_ss = 0.0; self.ar_n = 0
        self.arbuf = deque(maxlen=self.p)
        self.roll = {w: (deque(maxlen=w), [0.0, 0.0]) for w in self.rw}
        self.ew = {a: self.mu_h for a in self.alphas}
        self.ewv = {a: 0.0 for a in self.alphas}
        return self

    def update(self, x):
        x = float(x)
        self.n += 1; n = self.n
        sd_h = self.sd_h; mu_h = self.mu_h; var_h = self.var_h
        # --- Welford up to 4th moment ---
        d = x - self.mean; dn = d / n; dn2 = dn * dn
        term = d * dn * (n - 1)
        self.mean += dn
        self.M4 += term * dn2 * (n * n - 3 * n + 3) + 6 * dn2 * self.M2 - 4 * dn * self.M3
        self.M3 += term * dn * (n - 2) - 3 * dn * self.M2
        self.M2 += term
        var = self.M2 / (n - 1) if n > 1 else 0.0
        std = math.sqrt(var) if var > 0 else 0.0
        # --- standardized point vs reference ---
        zx = (x - mu_h) / sd_h
        # --- cumsum range + two-sided CUSUM ---
        self.cumsum += zx
        if self.cumsum > self.cmax: self.cmax = self.cumsum
        if self.cumsum < self.cmin: self.cmin = self.cumsum
        cp = self.cusum_p + zx - 0.5
        cn = self.cusum_n - zx - 0.5
        self.cusum_p = cp if cp > 0.0 else 0.0
        self.cusum_n = cn if cn > 0.0 else 0.0
        if self.cusum_p > self.cusum_max: self.cusum_max = self.cusum_p
        if self.cusum_n > self.cusum_max: self.cusum_max = self.cusum_n
        # --- tails ---
        azx = zx if zx >= 0 else -zx
        if azx > 2.0: self.tail2 += 1
        if azx > 3.0: self.tail3 += 1
        # --- jumps / sign runs ---
        if self.prev is not None:
            if abs(x - self.prev) > 2.0 * sd_h: self.jumps += 1
            if (x - mu_h) * (self.prev - mu_h) < 0: self.signruns += 1
            self.sxy += x * self.prev
        self.sx += x; self.sxx += x * x
        # --- rank vs reference (Mann-Whitney) ---
        lo = bisect_left(self.sorted_ref, x); hi = bisect_right(self.sorted_ref, x)
        self.rank_sum += 0.5 * (lo + hi) * self._inv_refn
        # --- KS histogram (equal-freq ref bins) ---
        self.binc[bisect_right(self.edges, x)] += 1
        # --- AR residual vs reference-fit AR ---
        if self.ar is not None and len(self.arbuf) == self.p:
            lag = np.array(self.arbuf, dtype=np.float64)[::-1]
            pred = self.ar[0] + self.ar[1:] @ lag
            r_ = x - pred; self.ar_resid_ss += r_ * r_; self.ar_n += 1
        self.arbuf.append(x)
        # --- EWMA mean/var ---
        for a in self.alphas:
            m0 = self.ew[a]
            self.ew[a] = (1 - a) * m0 + a * x
            self.ewv[a] = (1 - a) * (self.ewv[a] + a * (x - m0) ** 2)
        # --- rolling windows via ring buffer ---
        for w, (buf, acc) in self.roll.items():
            if len(buf) == buf.maxlen:
                old = buf[0]; acc[0] -= old; acc[1] -= old * old
            buf.append(x); acc[0] += x; acc[1] += x * x
        self.prev = x

        # --- emit into preallocated buffer -----------------------------
        out = self._out
        se = sd_h / math.sqrt(n)
        emp = np.cumsum(self.binc) / n
        ks = float(np.max(np.abs(emp - self._bin_targets)))
        if n > 2 and var > 0:
            mx = self.sx / n
            cov = self.sxy / (n - 1) - mx * mx * n / (n - 1)
            ac1 = cov / var
        else:
            ac1 = 0.0
        vg = n >= self.VAR_MIN
        sqrt_n = math.sqrt(n)
        out[0]  = math.log1p(n)
        out[1]  = (self.mean - mu_h) / max(se, 1e-8)
        out[2]  = zx
        out[3]  = math.log((var + 1e-8) / var_h) if vg else 0.0
        out[4]  = std / sd_h
        out[5]  = (self.M3 / n) / (var ** 1.5 + 1e-8) if vg else 0.0
        out[6]  = ((self.M4 / n) / (var ** 2 + 1e-8) - 3.0) if vg else 0.0
        out[7]  = (self.cmax - self.cmin) / sqrt_n
        out[8]  = self.cusum_max / sqrt_n
        out[9]  = self.tail2 / n
        out[10] = self.tail3 / n
        out[11] = self.jumps / n
        out[12] = self.signruns / n
        out[13] = ks
        out[14] = self.rank_sum / n - 0.5
        out[15] = ac1
        out[16] = ac1 - self.ref_ac1
        out[17] = (math.log((self.ar_resid_ss / self.ar_n + 1e-8)
                            / (self.ar_var_h + 1e-8))
                   if self.ar_n >= self.VAR_MIN else 0.0)
        out[18] = self.ref_log_n
        out[19] = self.ref_skew
        out[20] = self.ref_kurt
        out[21] = self.ref_ac1
        # EWMA block
        off = self._ew_off
        for ai, a in enumerate(self.alphas):
            ew_se = sd_h * math.sqrt(a / (2 - a))
            out[off + 2 * ai]     = (self.ew[a] - mu_h) / max(ew_se, 1e-8)
            out[off + 2 * ai + 1] = math.log((self.ewv[a] + 1e-8) / var_h) if vg else 0.0
        # Rolling-window block
        off = self._roll_off
        for wi, w in enumerate(self.rw):
            buf, acc = self.roll[w]
            L = len(buf)
            m = acc[0] / L
            v = acc[1] / L - m * m
            if v < 0.0: v = 0.0
            out[off + 2 * wi]     = (m - mu_h) / sd_h
            out[off + 2 * wi + 1] = math.log((v + 1e-8) / var_h)
        return out


# ---- training subsampling ------------------------------------------------
def _log_steps(L, m):
    if L <= m:
        return np.arange(L)
    return np.unique(
        np.round(np.expm1(np.linspace(0, np.log1p(L - 1), m))).astype(int)
    )


# ---- train / infer -------------------------------------------------------
def train(datasets: List[Tuple[int, List[float], List[float], Optional[int]]],
          model_directory_path: str):
    params = dict(objective='binary', n_estimators=500, learning_rate=0.05,
                  num_leaves=63, min_child_samples=300, subsample=0.8,
                  subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                  n_jobs=1, verbosity=-1)
    samples_per = 40   # log-spaced online steps sampled per series
    n_seeds = 1        # single booster: 3x faster infer, ~-0.005 TS-AUC

    st = StreamState()
    cols = st.feature_names
    rows, labels, sids = [], [], []
    for dataset_id, x_hist, x_online, tau in datasets:
        xo = np.asarray(x_online, dtype=np.float64)
        st.reset(x_hist); L = len(xo)
        want = set(_log_steps(L, samples_per).tolist())
        for k in range(L):
            out = st.update(xo[k])
            if k not in want:
                continue
            rows.append(out.copy())  # buffer is shared — must copy
            labels.append(1 if (tau is not None and k >= tau) else 0)
            sids.append(dataset_id)

    Xtr = np.asarray(rows, np.float32); ytr = np.asarray(labels, np.int8)
    sids = np.asarray(sids)
    # equal-series sample weights (the +0.02 win from the campaign)
    cmap = dict(zip(*np.unique(sids, return_counts=True)))
    w = np.array([1.0 / cmap[s] for s in sids]); w /= w.mean()
    models = [lgb.LGBMClassifier(random_state=s, **params).fit(Xtr, ytr, sample_weight=w)
              for s in range(n_seeds)]
    boosters = [m.booster_ for m in models]
    joblib.dump({'boosters': boosters, 'cols': cols},
                os.path.join(model_directory_path, 'model.joblib'))
    print('trained: %d rows, %d feats, %d series' % (len(Xtr), len(cols), len(cmap)))


def infer(datasets: Iterable[Tuple[List[float], Iterable[float]]],
          model_directory_path: str):
    
    m = joblib.load(os.path.join(model_directory_path, 'model.joblib'))
    boosters, cols = m['boosters'], m['cols']
    # NOTE: no reset_parameter / params-dict mutation on joblib-loaded boosters —
    # that combo can corrupt state and segfault under parallel workers.
    # num_threads is passed per-call via predict(..., num_threads=1) below.

    st = StreamState()
    if st.feature_names != cols:
        raise RuntimeError("feature schema drift: train vs infer StreamState differ")
    ncol = st.ncol
    n_boost = len(boosters)
    vec = np.empty((1, ncol), dtype=np.float64)
    inv_nb = 1.0 / n_boost

    yield  # signal readiness

    for x_historical, x_online in datasets:
        st.reset(x_historical)
        for point in x_online:
            out = st.update(point)
            vec[0, :] = out                # 30-float copy, cheap
            s = 0.0
            for b in boosters:
                s += b.predict(vec, num_threads=1)[0]
            yield float(s * inv_nb)

In [15]:
# Keep-block: exported verbatim to the cloud script.
# Parallel workers fork after LightGBM's OpenMP is initialized in the parent —
# that's what segfaulted the 4-worker run. Single worker is fine within budget.
# @crunch/keep:on
INFER_PARALLELISM = 1

# @crunch/keep:off

In [ ]:
# Local test + submit — MUST stay outside any @crunch/keep block, else the cloud
# script imports crunch_tools and crashes with NameError.

# from sklearn.metrics import roc_auc_score
# crunch_tools.test()
# prediction = pd.read_parquet('prediction/prediction.parquet')
# y_test = pd.read_parquet('data/y_test.reduced.parquet')
# m = prediction.merge(y_test, how='left', left_index=True, right_index=True)
# m['t'] = m.groupby('id').cumcount()
# num = den = 0.0
# for t, g in m.groupby('t'):
#     yv = g['target'].values; sv = g['prediction'].values
#     npos = int(yv.sum()); nneg = len(yv) - npos
#     if npos == 0 or nneg == 0: continue
#     num += npos * nneg * roc_auc_score(yv, sv); den += npos * nneg
# print('Local TS-AUC: %.4f' % (num / den if den else 0.5))

# crunch_tools.submit(message='streaming GBM — num_threads=1 + ndarray emit')

11:46:57 forbidden library: deque (python)
11:46:57   -> request to whitelist: https://hub.crunchdao.com/competitions/structural-break-real-time/resources/whitelisted-libraries?requestAlias=deque&requestLanguage=PYTHON
11:46:57 note: detection is based on the previous execution of the cell; if you have already corrected the issue(s), please restart the kernel and run all cells again
11:46:57 
11:46:58 started
11:46:58 running local test
11:46:58 internet access isn't restricted, no check will be done
11:46:58 
11:46:58 executing - command=train


data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_train.parquet (218514418 bytes)
data/X_train.parquet: already exists, file length match
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_test.reduced.parquet (2587435 bytes)
data/X_test.reduced.parquet: already exists, file length match
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_train.parquet (8356193 bytes)
data/y_train.parquet: already exists, file length match
data/y_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_test.reduced.parquet (106299 bytes)
data/y_test.reduced.parquet: already exists, file length match
data/y_test_index.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws

11:48:39 executing - command=get_parallelism
11:48:39 using a parallelism of 1
11:48:39 executing - command=infer


trained: 328996 rows, 30 feats, 10000 series


11:48:44 checking determinism by executing the inference again with 10% of the data (tolerance: 1e-08)
11:48:44 executing - command=infer
11:48:44 save prediction - path=prediction
11:48:44 determinism check: passed
11:48:44 ended
11:48:44 
11:48:44 duration - time=00:01:46
11:48:44 memory - before="231.37 MB" after="427.08 MB" consumed="195.71 MB"


Local TS-AUC: 0.5188



---

Your work could not be submitted automatically, please do so manually:
1. Download your Notebook from Colab
2. Upload it to the platform
3. Create a run to validate it

### >> [https://hub.crunchdao.com/competitions/structural-break-real-time/submit/notebook](https://hub.crunchdao.com/competitions/structural-break-real-time/submit/notebook?projectName=submission1&message=streaming+GBM+%E2%80%94+num_threads%3D1+%2B+ndarray+emit)

<img alt="Download and Submit Notebook" src="https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/animations/download-and-submit-notebook.gif" height="600px" />

<br />
<small>Error preventing submit: <code>No module named 'google'</code></small>
